In [8]:
import pandas as pd
from pathlib import Path
captions = pd.read_json('/home/nprox/AML-DCGAN/yfcc100m_captioned.jsonl', lines=True)
captions['stem'] = captions['filepath'].map(Path).map(lambda x: x.stem)
df = pd.read_json('/home/nprox/datasets/yfcc100m_NL_captions.json')
df['stem'] = df['local_path'].map(Path).map(lambda x: x.stem)
display((lambda x: x[x['captions'].notna()])(pd.merge(df, captions[['stem', 'captions']], on='stem', how='left')))

,local_path,photo_id,caption,processed,stem,captions
196,/mnt/data/yfcc100m/26c/c3b/26cc3bbcdb7d7563133...,4605398871,The wall at the end of the path that goes nowh...,False,26cc3bbcdb7d756313327558975baf3,{'small': 'A long red brick wall stands behind...
2984,/mnt/data/yfcc100m/1ef/73a/1ef73a2f7f39a95eec1...,5376334027,Costa Rica - VBT Highlands and Coast. VBT Walk...,False,1ef73a2f7f39a95eec1d7efce52f3dbd,{'small': 'A scenic coastal view features a ca...
3195,/mnt/data/yfcc100m/26a/b34/26ab3429af0b57cac10...,5558545636,Fondren After 5. The art on display in Brown's...,False,26ab3429af0b57cac10ad1c15f45868,{'small': 'A blonde woman in a black fur coat ...
3290,/mnt/data/yfcc100m/dcb/4e9/dcb4e9e4ea0f3b4adae...,6623674565,Santa's coming!...big snow of December 2010...,False,dcb4e9e4ea0f3b4adaefec5c26ac67d,{'wild': 'A LEGO Santa Claus figure stands in ...
3565,/mnt/data/yfcc100m/a9a/7e6/a9a7e634c197fac12c3...,3183982448,Grillparty. The grill.,False,a9a7e634c197fac12c3cf1b253dd32b9,{'small': 'A man in a red shirt cooks food on ...
...,...,...,...,...,...,...
8694775,/mnt/data/yfcc100m/c16/9d2/c169d2cc2577c6451d1...,6465357021,"President Van Rompuy talks to Nikola Gruevski,...",False,c169d2cc2577c6451d1a53c1d5cc1280,{'small': 'Two men in suits sit in armchairs c...
8694991,/mnt/data/yfcc100m/f55/7c8/f557c875f633b6caebc...,54227323,New work uniform. This is my new work uniform:...,False,f557c875f633b6caebc06641729e35ac,"{'small': 'A smiling man wears a black ""Lomba ..."
8695608,/mnt/data/yfcc100m/5af/f33/5aff334f8cb5fd9977c...,6345432874,Greene King Brewery Tour 28-07-2011. Tour the ...,False,5aff334f8cb5fd9977cfa8d5ea27fbdd,{'wild': 'A person in historical costume kneel...
8695887,/mnt/data/yfcc100m/b95/a41/b95a41a151261edf9c7...,291875673,First snow on the Galen mountain. First snow o...,False,b95a41a151261edf9c7b1f34e4672035,{'small': 'A snow-capped mountain rises majest...


In [51]:
import pandas as pd
from pathlib import Path
data = pd.read_csv('//home//nprox//datasets//yfcc100m_subset_data.tsv', names=['line_number', 'photo_id', 'photo_hash'], header=None, sep='\t')
folders = data['photo_hash'].map(lambda x: x[:3]).unique()

drives_dir = Path('/mnt') #Path(args.drives_dir)
path_map = {fldr.name: Path(fldr) 
            for drive in drives_dir.iterdir() 
            for fldr in (drive/'yfcc100m').iterdir() 
            if fldr.name in folders}
# all photos .jpg
dwnld_photos = data['photo_hash'].map(lambda x: path_map.get(x[:3], Path())/Path(x[3:6])/Path(x+'.jpg') if path_map.get(x[:3], False) != False else pd.NA).dropna()
dwnld_photos = pd.DataFrame(dwnld_photos.to_numpy(), columns=['local_path'])
#dwnld_photos = dwnld_photos[dwnld_photos.map(Path.exists)] #.map(lambda x: x if x.exists() else pd.NA).dropna()
dwnld_photos['photo_hash'] = dwnld_photos['local_path'].map(lambda x: x.stem)
dwnld_photos = pd.merge(dwnld_photos, data[['photo_hash', 'photo_id']], on='photo_hash', how='left')
"""
dwnld = [fldr.name for drive in drives_dir.iterdir() 
        for fldr in (drive/'yfcc100m').iterdir() if fldr.name in folders]
to_dwnld = [fldr for fldr in folders if fldr not in dwnld]"""


"\ndwnld = [fldr.name for drive in drives_dir.iterdir() \n        for fldr in (drive/'yfcc100m').iterdir() if fldr.name in folders]\nto_dwnld = [fldr for fldr in folders if fldr not in dwnld]"

In [77]:
dwnld_photos['photo_hash'].str[:3].unique()

<StringArray>
['b61', '4be', 'faa', 'ee4', 'bb6', 'a42', 'bd6', '4aa', 'de8', '47b',
 ...
 '06f', '009', '0ae', '08d', '00a', '00e', '005', '003', '00c', '001']
Length: 2359, dtype: str

In [3]:
import sqlite3

con=sqlite3.connect('file:/home/nprox/datasets/yfcc100m_dataset.sql?mode=ro',uri=True)
con.execute('''
    CREATE TEMP TABLE IF NOT EXISTS photoid (
        photoid INTEGER PRIMARY KEY
    )
''')
con.execute('DELETE FROM temp.photoid')

con.executemany(
    'INSERT OR IGNORE INTO temp.photoid(photoid) VALUES (?)',
    ((int(x),) for x in dwnld_photos['photo_id'])
)
con.commit()

df = pd.read_sql_query('''
    SELECT y.photoid, y.title, y.description
    FROM temp.photoid AS w
    JOIN main.yfcc100m_dataset AS y ON y.photoid = w.photoid
''', con)

In [ ]:
from urllib.parse import unquote_plus
from bs4 import BeautifulSoup

dwnld_photos = (lambda x: x[[y for y in x.columns if y != 'photoid']])(pd.merge(dwnld_photos, df, left_on='photo_id', right_on='photoid'))
dwnld_photos['title'] = dwnld_photos['title'].map(unquote_plus)
dwnld_photos['description'] = dwnld_photos['description'].map(unquote_plus)
dwnld_photos.loc[dwnld_photos['description'].str.contains('<|>', regex=True), 'description'] = \
    dwnld_photos[dwnld_photos['description'].str.contains('<|>', regex=True)]['description'].map(lambda x: BeautifulSoup(x, 'lxml').get_text(' ', strip=True), na_action='ignore')
dwnld_photos.loc[~dwnld_photos['title'].str.contains(r'\.$'), 'title'] = dwnld_photos[~dwnld_photos['title'].str.contains(r'\.$')]['title'] + '.'
dwnld_photos['title'] = dwnld_photos['title'].str.replace(r'\.+\W*', '. ')
dwnld_photos['caption'] = dwnld_photos['title'] + dwnld_photos['description']
dwnld_photos

,local_path,photo_hash,photo_id,title,description
0,/mnt/data/yfcc100m/b61/f25/b61f252678da02eaff4...,b61f252678da02eaff453bce575cb6,2297552664,denise's peanut chicken,recipe here: allrecipes.com/Recipe/Denises-Pea...
1,/mnt/data/yfcc100m/4be/1ac/4be1ac2e9425d56e914...,4be1ac2e9425d56e9140baa8457c399,2932067831,spot the intrepid explorer,The Arches National Park is a wonderland of ge...
2,/mnt/data/yfcc100m/faa/524/faa524f25942ba9362d...,faa524f25942ba9362db7495a61651c,5177649120,Yau Choy,Review: Chong Yen (Steam Fish) Food Stall
3,/mnt/data2/yfcc100m/ee4/2a0/ee42a0b73fc6753b87...,ee42a0b73fc6753b877131e84b39fb,9743459054,ME White High Collar Thobe 1,ME White High Collar Thobe
4,/mnt/data/yfcc100m/bb6/8d7/bb68d7fee09d1ef0837...,bb68d7fee09d1ef0837eb3a4d2c8ab8c,2245962407,The Fight (step 3),Stars Vs. Canucks - 2/5/08 - the Stars and Can...
...,...,...,...,...,...
8561554,/mnt/data/yfcc100m/364/31c/36431c5c8342a26a29d...,36431c5c8342a26a29de28a5ec98926e,5786441045,The Geysers - Tour in February 2011,A tour to the Geysers geothermal field in Cali...
8561555,/mnt/data/yfcc100m/3d8/739/3d8739fa25813e32659...,3d8739fa25813e3265958fd9437f58,4506571174,Ravine April 9 - Afternoon Light,"Part of the set ""Ravine Views"" - there is a ra..."
8561556,/mnt/data2/yfcc100m/19d/5f8/19d5f84ebf86d3d31a...,19d5f84ebf86d3d31ae3e2c094a1733d,7179040256,The Waldos,30th Anniversary at Bowery Electric. Featuring...
8561557,/mnt/data2/yfcc100m/739/2df/7392df6fd6a3a37ac7...,7392df6fd6a3a37ac7b9bd517a591e,4899784138,My Grandfather,Photos taken by my dad at my graduation (from ...


In [76]:
import os
str(Path(os.getcwd()))

'/home/nprox/AML-DCGAN'

In [71]:
dwnld_photos

,local_path,photo_hash,photo_id,title,description,caption
0,/mnt/data/yfcc100m/b61/f25/b61f252678da02eaff4...,b61f252678da02eaff453bce575cb6,2297552664,denise's peanut chicken.,recipe here: allrecipes.com/Recipe/Denises-Pea...,denise's peanut chicken. recipe here: allrecip...
1,/mnt/data/yfcc100m/4be/1ac/4be1ac2e9425d56e914...,4be1ac2e9425d56e9140baa8457c399,2932067831,spot the intrepid explorer.,The Arches National Park is a wonderland of ge...,spot the intrepid explorer. The Arches Nationa...
2,/mnt/data/yfcc100m/faa/524/faa524f25942ba9362d...,faa524f25942ba9362db7495a61651c,5177649120,Yau Choy.,Review: Chong Yen (Steam Fish) Food Stall,Yau Choy. Review: Chong Yen (Steam Fish) Food ...
3,/mnt/data2/yfcc100m/ee4/2a0/ee42a0b73fc6753b87...,ee42a0b73fc6753b877131e84b39fb,9743459054,ME White High Collar Thobe 1.,ME White High Collar Thobe,ME White High Collar Thobe 1. ME White High Co...
4,/mnt/data/yfcc100m/bb6/8d7/bb68d7fee09d1ef0837...,bb68d7fee09d1ef0837eb3a4d2c8ab8c,2245962407,The Fight (step 3).,Stars Vs. Canucks - 2/5/08 - the Stars and Can...,The Fight (step 3). Stars Vs. Canucks - 2/5/08...
...,...,...,...,...,...,...
8561554,/mnt/data/yfcc100m/364/31c/36431c5c8342a26a29d...,36431c5c8342a26a29de28a5ec98926e,5786441045,The Geysers - Tour in February 2011.,A tour to the Geysers geothermal field in Cali...,The Geysers - Tour in February 2011. A tour to...
8561555,/mnt/data/yfcc100m/3d8/739/3d8739fa25813e32659...,3d8739fa25813e3265958fd9437f58,4506571174,Ravine April 9 - Afternoon Light.,"Part of the set ""Ravine Views"" - there is a ra...",Ravine April 9 - Afternoon Light. Part of the ...
8561556,/mnt/data2/yfcc100m/19d/5f8/19d5f84ebf86d3d31a...,19d5f84ebf86d3d31ae3e2c094a1733d,7179040256,The Waldos.,30th Anniversary at Bowery Electric. Featuring...,The Waldos. 30th Anniversary at Bowery Electri...
8561557,/mnt/data2/yfcc100m/739/2df/7392df6fd6a3a37ac7...,7392df6fd6a3a37ac7b9bd517a591e,4899784138,My Grandfather.,Photos taken by my dad at my graduation (from ...,My Grandfather. Photos taken by my dad at my g...


In [70]:
dwnld_photos['caption'] = dwnld_photos['title'] + " " + dwnld_photos['description']

In [48]:
from bs4 import BeautifulSoup

dwnld_photos.loc[dwnld_photos['description'].str.contains('<|>', regex=True), 'description'] = dwnld_photos[dwnld_photos['description'].str.contains('<|>', regex=True)]['description'].map(lambda x: BeautifulSoup(x, 'lxml').get_text(' ', strip=True), na_action='ignore')
dwnld_photos

,local_path,photo_hash,photo_id,title,description
0,/mnt/data/yfcc100m/b61/f25/b61f252678da02eaff4...,b61f252678da02eaff453bce575cb6,2297552664,denise's peanut chicken,recipe here: allrecipes.com/Recipe/Denises-Pea...
1,/mnt/data/yfcc100m/4be/1ac/4be1ac2e9425d56e914...,4be1ac2e9425d56e9140baa8457c399,2932067831,spot the intrepid explorer,The Arches National Park is a wonderland of ge...
2,/mnt/data/yfcc100m/faa/524/faa524f25942ba9362d...,faa524f25942ba9362db7495a61651c,5177649120,Yau Choy,Review: Chong Yen (Steam Fish) Food Stall
3,/mnt/data2/yfcc100m/ee4/2a0/ee42a0b73fc6753b87...,ee42a0b73fc6753b877131e84b39fb,9743459054,ME White High Collar Thobe 1,ME White High Collar Thobe
4,/mnt/data/yfcc100m/bb6/8d7/bb68d7fee09d1ef0837...,bb68d7fee09d1ef0837eb3a4d2c8ab8c,2245962407,The Fight (step 3),Stars Vs. Canucks - 2/5/08 - the Stars and Can...
...,...,...,...,...,...
8561554,/mnt/data/yfcc100m/364/31c/36431c5c8342a26a29d...,36431c5c8342a26a29de28a5ec98926e,5786441045,The Geysers - Tour in February 2011,A tour to the Geysers geothermal field in Cali...
8561555,/mnt/data/yfcc100m/3d8/739/3d8739fa25813e32659...,3d8739fa25813e3265958fd9437f58,4506571174,Ravine April 9 - Afternoon Light,"Part of the set ""Ravine Views"" - there is a ra..."
8561556,/mnt/data2/yfcc100m/19d/5f8/19d5f84ebf86d3d31a...,19d5f84ebf86d3d31ae3e2c094a1733d,7179040256,The Waldos,30th Anniversary at Bowery Electric. Featuring...
8561557,/mnt/data2/yfcc100m/739/2df/7392df6fd6a3a37ac7...,7392df6fd6a3a37ac7b9bd517a591e,4899784138,My Grandfather,Photos taken by my dad at my graduation (from ...


In [44]:
dwnld_photos[~dwnld_photos['description'].str.contains('<')].shape[0] - ((lambda x: x[~x.str.contains('<')])(dwnld_photos['description']).map(unescape)).shape[0]

0

In [35]:
dwnld_photos['description'].map(lambda x: x if '<' not in x else BeautifulSoup(x, 'lxml').get_text(' ', strip=True), na_action='ignore')

KeyboardInterrupt: 

In [33]:
from html import unescape
from bs4 import BeautifulSoup
(lambda x: x[~x[0].str.contains('<a href="(?:.*rel="nofollow")?|^<.?.?>|^</?del>|^</?strong>')][0].map(lambda x: BeautifulSoup(unescape(x), 'lxml').get_text(' ', strip=True)))(dwnld_photos['description'].str.extractall(r'(<.*>)').dropna())

         match
1035     0                            http://ClintJCL.wordpress.com
1491     0                                         Sissel - One Day
2496     0            http://www.day-trips.ca/content/niagara-falls
2511     0                            http://ClintJCL.wordpress.com
2937     0                                       Read the Blog Post
                                        ...                        
8560931  4                                                 Part III
         5        Bradley Bodiford, a Pi Kappa Phi at Mercer Uni...
         6        Josh Rhodes explains Pi Kappa Phi's Build Amer...
8561175  0                                                 On black
8561285  0                                       PopCultureGeek.com
Name: 0, Length: 16045, dtype: str

In [ ]:
(lambda x: (lambda y: y[~y[0].str.contains('<..?>', regex=True)])(x[~x[0].str.contains('<a href="')]))(dwnld_photos['description'].str.extractall(r'(<.*>)').dropna())

0
        match                                                   
667     0      <strong>More photos, videos e the setlist: </s...
1685    0      <del>The museum was closing when I took this p...
2514    1           <strong>Find The Cosmopolitan on...</strong>
3994    0                                      <strong></strong>
4521    2                                               <strong>
...                                                          ...
8555516 7                              <strong>Library:</strong>
8556310 0                           <strong>the shot...</strong>
        1                          <strong>and then... </strong>
        2                          <strong>photoshop...</strong>
8557507 0       <strong>New Travel Guide and Photo-map:</strong>

[16593 rows x 1 columns]

In [ ]:

"""
con.execute(
 'SELECT photoid, title,description FROM yfcc100m_dataset WHERE photoid=?',
 (int(dwnld_photos['photo_id'].iloc[234]),)
).fetchall()
"""
df = pd.DataFrame(con.execute(f'SELECT photoid, title,description FROM yfcc100m_dataset').fetchall(), columns=['photo_id', 'title', 'description'])

In [ ]:
dwnld_photos.iloc[:10000].map(lambda x: next(x.parent.glob(f'{x.stem}*'), "None")).dropna()

0        /mnt/data/yfcc100m/b61/f25/b61f252678da02eaff4...
2        /mnt/data/yfcc100m/4be/1ac/4be1ac2e9425d56e914...
3        /mnt/data/yfcc100m/faa/524/faa524f25942ba9362d...
4        /mnt/data2/yfcc100m/ee4/2a0/ee42a0b73fc6753b87...
5        /mnt/data/yfcc100m/bb6/8d7/bb68d7fee09d1ef0837...
                               ...                        
17570    /mnt/data/yfcc100m/183/23c/18323cae5642cc667e7...
17571    /mnt/data/yfcc100m/3d8/eab/3d8eabf1c832d892fff...
17572    /mnt/data/yfcc100m/797/9f1/7979f1329561ae78197...
17573    /mnt/data2/yfcc100m/86a/538/86a538c78e7ad36814...
17574    /mnt/data2/yfcc100m/77f/be0/77fbe06f28151fdbf3...
Name: photo_hash, Length: 9997, dtype: object

In [60]:
path_map = {fldr.name: Path(fldr) 
            for drive in drives_dir.iterdir() 
            for fldr in (drive/'yfcc100m').iterdir() 
            if fldr.name in folders}
dwnld_photos = data['photo_hash'].map(lambda x: next((path_map.get(x[:3], Path())/Path(x[3:6])).glob(f'{x}*'), None) if path_map.get(x[:3], False) != False else pd.NA).dropna()

KeyboardInterrupt: 

In [26]:
import os
dwnld_photos.map(lambda x: os.path.exists(x)).value_counts()

KeyboardInterrupt: 

In [37]:
display(len({1:2, 3:4}))
raise Exception('sdjnclksdm')

2

Exception: sdjnclksdm

In [ ]:
from pathlib import Path
import numpy as np
import subprocess
import argparse
import shutil
import ollama
import json
import sys
import re
import os

# %pip install git+https://github.com/openai/CLIP.git
#import clip
#from clip.simple_tokenizer import SimpleTokenizer
"""
parser = argparse.ArgumentParser(description=__doc__)
parser.add_argument("--drives-dir", type=Path, default="/mnt")
parser.add_argument("--out-dir", type=Path, default=os.getcwd())
parser.add_argument("--parallel", type=int, default=18)
parser.add_argument("--model", type=str, default='qwen3.8-27b:q4_k_m')
args = parser.parse_args()"""

yfcc100m = []
drives_dir = Path('/mnt') #Path(args.drives_dir)
for drive in drives_dir.iterdir():
    drive_dir = drives_dir / re.match(r'.*/([^/]+)$', str(drive)).group(1)
    if not os.path.exists(drive_dir / 'yfcc100m'):
        continue
    drive_dir = drive_dir / 'yfcc100m'
    yfcc100m.extend([Path(sub_fldr) for fldr in drive_dir.iterdir() for sub_fldr in fldr.iterdir() \
                     if (fldr/'.complete').is_file() and not (sub_fldr/'.parsed').is_file() and \
                        not re.match(r'.*/.complete$', str(sub_fldr))])
indices = np.random.permutation(np.arange(yfcc100m.__len__()))

PROMPT = ["Describe this image in one concise English sentence, at most 25 words. Mention the main subject, action, and setting when visible. Return only the caption.", \
          "Return only a caption",]
subprocess.run(['llama-server', f'-hf bartowski/{args.model}}', '--n-gpu-layers 99', f'--parallel {args.parallel}', 
            '--ctx-size 8192', '--cont-batching', '--jinja', '--reasoning off', '--alias yfcc-captioner', 
            '--host 127.0.0.1', '--port 8080'], capture_output=False, text=True, check=True,)


# batch then add to whole
fldr = Path(os.getcwd()) # Path(args.out_dir)
for idx in indices:
    batch = ''
    for img in Path(yfcc100m[idx]).iterdir():
        # construct json in mem? time diff?
        if not img.is_file() or img.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
            continue

        response = ollama.chat(
            model=args.model,
            messages=[{
                "role": "user",
                "content": PROMPT,
                "images": [img.read_bytes()],
            }],
            think=False,
            stream=False,
            keep_alive="30m",
            options={
                "num_ctx": 4096,
                "num_predict": 77,
                "temperature": 0.2,
            },
        )

        caption = response.message.content.strip()
        if not caption or response.done_reason == "length":
            raise RuntimeError(f"Empty or truncated caption for {img}; request a shorter caption.")
        batch += caption
    (yfcc100m['idx']/'.parsed').touch()
    (fldr/'yfcc100m_captioned.jsonl').touch()
    with (fldr/'yfcc100m_captioned.jsonl').open("a", encoding="utf-8") as file:
        file.write(json.dumps(batch, ensure_ascii=False, default=str) + "\n")


/mnt/data/yfcc100m/8f3/7e0/8f37e055bded36339ef06ca164a44bbd.jpg
/mnt/data/yfcc100m/8f3/7e0/8f37e017ab4c5f3f3e27ec705d263675.jpg
/mnt/data/yfcc100m/8f3/7e0/8f37e0a85b959c51ae8d24cfb246b3a.jpg
/mnt/data/yfcc100m/8f3/7e0/8f37e0bece8ce226f588fe680268cfd.jpg
/mnt/data/yfcc100m/8f3/7e0/8f37e0254b2b31d7d4f4adae27dbde4.jpg
/mnt/data/yfcc100m/8f3/7e0/8f37e037ee32867bac39ab706e5a66e7.jpg
/mnt/data/yfcc100m/8f3/7e0/8f37e034ffa3ea63139c34cd7a4bab52.jpg
/mnt/data/yfcc100m/8f3/7e0/8f37e01af858a17851ebbc645dfdd3.jpg


In [ ]:


fldr = Path.cwd()  # Or Path(args.out_dir)
output = fldr / "yfcc100m_captioned.jsonl"

for idx in indices:
    batch = fldr / "batch.jsonl"
    # Reset the temporary batch and keep it open for this subfolder.
    with batch.open("w", encoding="utf-8") as file:
        for img in source.iterdir():
            

            value = {"image": str(img), "caption": caption}
            file.write(json.dumps(value, ensure_ascii=False) + "\n")

    # Each batch line is already JSON; append its text directly.
    with batch.open("r", encoding="utf-8") as src, \
         output.open("a", encoding="utf-8") as dst:
        shutil.copyfileobj(src, dst)

    (source / ".parsed").touch()

In [35]:
MODEL = "qwen3.8:27b-q4_k_m"
PROMPT = (
    "Describe this image in one concise English sentence, at most 25 words. "
    "Mention the main subject, action, and setting when visible. "
    "Return only the caption."
)

response = ollama.chat(
                model=MODEL,
                messages=[{
                    "role": "user",
                    "content": 'return only a caption', #PROMPT,
                    "images": [Path('/mnt/data/yfcc100m/8f3/7e0/8f37e01af858a17851ebbc645dfdd3.jpg').read_bytes()],
                }],
                think=False,
                stream=False,
                keep_alive="30m",
                options={
                    "num_ctx": 4096,
                    "num_predict": 77,
                    "temperature": 0.2,
                },
            )

caption = response.message.content.strip()
caption

'A smiling woman with short dark hair, wearing a white long-sleeved athletic top and black shorts, stands indoors with her arms crossed, turning to look over her shoulder at the camera. She has a race bib or event tag pinned to the back of her shirt. The room features light green walls with wood paneling, a wooden chair, and a gray upholstered armchair.'

In [29]:
yfcc100m.__len__()

237486

In [ ]:
for subfldr in Path('/mnt/data/yfcc100m/365').iterdir():
    if :
        print(subfldr)
yfcc100m.__len__()
os.getcwd()

/mnt/data/yfcc100m/365/.complete


'/home/nprox/AML-DCGAN'

In [9]:
%tb
raise Exception('sdjkfnvlkmfd')

SystemExit: 2

Exception: sdjkfnvlkmfd

In [ ]:
from pathlib import Path
import numpy as np
import subprocess
import shutil
import re
import os

yfcc100m = []
drives_dir = Path('/mnt/')
for drive in drives_dir.iterdir():
    drive_dir = drives_dir / re.match(r'.*/([^/]+)$', str(drive)).group(1)
    if not os.path.exists(drive_dir / 'yfcc100m'):
        os.mkdir(drive_dir / 'yfcc100m')
    drive_dir = drive_dir / 'yfcc100m'
    yfcc100m.extend([re.match(r'.*/([^/]+)$', str(fldr)).group(1) for fldr in drive_dir.iterdir()])

folders = subprocess.run(["aws", "s3", "ls", "s3://multimedia-commons/data/images/", "--no-sign-request", "--region", "us-west-2",], \
                        capture_output=True, text=True, check=True,).stdout.splitlines()
folders = [fdr for line in folders if (fdr := re.match(r".*PRE (.+)/", line).group(1)) not in yfcc100m]
indices = np.random.permutation(np.arange(folders.__len__()))

cntr = 0
for drive in drives_dir.iterdir():
    drive_dir = drives_dir / re.match(r'.*/([^/]+)$', str(drive)).group(1) / 'yfcc100m'
    for idx in indices[cntr:].copy():
        if shutil.disk_usage(drive_dir)[-1] < 5*1024**3:
            break
        print(folders[idx])
        dwnl = subprocess.run(["aws", 's3', 'sync', f's3://multimedia-commons/data/images/{folders[idx]}/',
                               (drive_dir/folders[idx]), '--no-sign-request', '--region', 'us-west-2',],)
        (drive_dir/folders[idx]).touch()
        cntr += 1

array([2311, 1657, 2197, ..., 2082, 1246,  798], shape=(4094,))

In [ ]:
dwnl = subprocess.run(["aws", 's3', 'sync', f's3://multimedia-commons/data/images/{folders[idx]}',
                        (drive_dir/folders[indices[100]]), '--no-sign-request', '--region', 'us-west-2',], \
                        text=True, check=True)

In [ ]:
np.array(folders)[idx]

array(['5d3', '194', '009', ..., 'c35', 'a78', '2e1'],
      shape=(4081,), dtype='<U3')

array(['000', '001', '002', ..., 'fee', 'fef', 'ff0'],
      shape=(4081,), dtype='<U3')